# PHS564 — Lecture 12: Target Trial Emulation I (Protocol Builder)
This notebook helps you convert a clinical question into a **target trial protocol**.

**Running example (you can replace):** early broad-spectrum antibiotics within 6h of ICU admission.

## Learning goals
1. Define eligibility, **time zero**, treatment strategies, outcome, follow-up.
2. Specify the causal contrast (ITT vs per-protocol) and effect measure (RD vs RR).
3. Run a structured **bias audit** (immortal time, selection, positivity).


In [ ]:
import textwrap
from pathlib import Path

def write_protocol_md(protocol: dict, out_path: str = 'protocol.md'):
    """Write protocol fields to a markdown file (simple, readable)."""
    p = protocol
    md = []
    md.append(f"# Target Trial Protocol — {p.get('title','(title)')}\n")
    for k in [
        'clinical_question','eligibility','time_zero','treatment_strategies',
        'assignment_procedure','follow_up','outcome','causal_contrast',
        'effect_measure','covariates_plan','analysis_plan','bias_audit'
    ]:
        md.append(f"## {k.replace('_',' ').title()}\n")
        val = p.get(k,'')
        if isinstance(val, (list, tuple)):
            for item in val:
                md.append(f"- {item}")
            md.append('')
        else:
            md.append(str(val).strip() + "\n")
    Path(out_path).write_text("\n".join(md), encoding='utf-8')
    return out_path


## 1) Protocol fields (fill these)
Fill the dictionary below. Keep each field **concrete** (operational definitions).

In [ ]:
protocol = {
    "title": "Early broad-spectrum antibiotics within 6 hours of ICU admission",
    "clinical_question": "Among adult ICU admissions with suspected sepsis, what is the effect of initiating broad-spectrum antibiotics within 6h (vs not) on 28-day mortality?",
    "eligibility": [
        "Age ≥ 18 at ICU admission",
        "First ICU stay for the hospital admission",
        "Suspected infection: blood cultures ordered within ±6h of ICU admit (proxy)",
        "Exclude: comfort-measures-only orders at baseline (if available)",
    ],
    "time_zero": "ICU admission time (intime) for the first ICU stay; baseline covariates measured before/intime.",
    "treatment_strategies": [
        "Strategy A (treated): start broad-spectrum IV antibiotics within 6h of ICU admit; allow changes thereafter.",
        "Strategy B (control): no broad-spectrum IV antibiotics initiated within 6h of ICU admit.",
    ],
    "assignment_procedure": "Observational emulation: define A=1 if antibiotic order time ≤ intime+6h; otherwise A=0.",
    "follow_up": "From ICU admit (time zero) to 28 days, death, discharge, or loss to follow-up; define administrative censoring at 28d.",
    "outcome": "All-cause mortality within 28 days of ICU admission (binary).",
    "causal_contrast": "Primary: per-protocol (effect of adhering to strategies through 6h window).",
    "effect_measure": "Risk difference (RD) at 28 days (primary); RR as secondary.",
    "covariates_plan": [
        "Baseline: age, sex, comorbidity score, admission type, baseline lactate (if available), SOFA components pre/intime.",
        "Avoid: vitals/labs after antibiotics initiation; avoid conditioning on post-baseline ICU interventions.",
    ],
    "analysis_plan": [
        "Primary estimator: IPW with logistic propensity model; stabilized weights; truncation at 1st/99th percentiles if needed.",
        "Diagnostics: propensity overlap plot; weight histogram; covariate balance table/plot.",
        "Robustness: alternate covariate set + no-truncation sensitivity.",
    ],
    "bias_audit": [
        "Immortal time risk: align time zero at ICU admit; define treatment within a fixed window; exclude outcomes before 6h if needed.",
        "Selection/censoring risk: discharge as censoring—consider IPCW if informative; report as limitation.",
        "Positivity/overlap risk: if near-deterministic treatment for high severity, restrict population or coarsen severity strata.",
        "Measurement/versions: antibiotic class heterogeneity—define broad-spectrum list; report as versions of treatment.",
    ],
}

In [ ]:
out_path = write_protocol_md(protocol, out_path='protocol.md')
print('Wrote:', out_path)

## 2) Bias audit (answer in 3–5 bullets each)
Write short, specific answers. Avoid vague statements like “confounding may exist.”

### A) Immortal time / time-zero alignment
- TODO

### B) Exchangeability plan
- What is your adjustment set and why?

### C) Positivity / overlap
- Where might overlap fail? What restriction would you use?

### D) Versions of treatment
- Are there meaningful versions? How will you handle/report them?


## 3) Deliverable checklist
- `protocol.md` generated and reviewed
- One diagram/timeline slide showing time zero and the treatment window
- One paragraph: primary estimand + effect measure
